# ArXiv Standard Inference Sweep
Comprehensive evaluation across 8 configurations (2 SigExt models x 2 Quantizations x 2 Shot types).

**Extension 1: Semantic Supervision** - Systematic comparison of model sizes and shot types.

**Metrics:**
- **BERT Score**: Semantic similarity (BERT-base-English)
- **ROUGE-1**: Lexical overlap
- **KIR**: Keyphrase Integration Ratio

## 1. Setup

In [1]:
# Install deps
!pip install -q transformers datasets accelerate bitsandbytes sentence-transformers \
    spacy rouge_score bert_score langchain langchain-community langchain-huggingface \
    huggingface_hub 'numpy<2.0' 'scipy>=1.10' matplotlib seaborn

!python -m spacy download en_core_web_sm

/usr/bin/bash: /home/marcantoniolopez/Documenti/github/projects/DNLPProj/.venv/bin/pip: /home/marcantoniolopez/Documenti/github/DNLPProj/.venv/bin/python3.12: interprete errato: File o directory non esistente
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 1.7 MB/s  0:00:07m0:00:0100:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import os
import warnings
import logging

# Suppress warning outputs
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import gc
import re
import torch
import json
import spacy
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    pipeline
)
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from huggingface_hub import login

# Load spacy for sentence segmentation
nlp = spacy.load("en_core_web_sm")

## 2. Configuration

In [3]:
# Auth
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except:
    HF_TOKEN = os.getenv("HF_TOKEN") or "YOUR_HF_TOKEN_HERE"

login(token=HF_TOKEN)

# Sweep configurations
SIGEXT_CONFIGS = {
    "1k-060t": {
        "model_id": "LookUpMark/sigext-arxiv-en-1k-060t",
        "skip_samples": 10000,
        "threshold": 0.60
    },
    "5k-060t": {
        "model_id": "LookUpMark/sigext-arxiv-en-5k-060t",
        "skip_samples": 10000,
        "threshold": 0.60
    }
}

QUANT_CONFIGS = {
    "4bit": {"load_in_4bit": True},
    "8bit": {"load_in_8bit": True}
}

SHOT_TYPES = ["zero-shot", "few-shot"]

GLOBAL_CONFIG = {
    "llm_model_id": "meta-llama/Llama-3.1-8B-Instruct",
    "num_test_samples": 100,
    "max_length": 2048,
    "output_dir": "./results"
}

os.makedirs(GLOBAL_CONFIG["output_dir"], exist_ok=True)
print(f"Sweep setup: {len(SIGEXT_CONFIGS)} models x {len(QUANT_CONFIGS)} quant x {len(SHOT_TYPES)} shot types = {len(SIGEXT_CONFIGS)*len(QUANT_CONFIGS)*len(SHOT_TYPES)} configs")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Sweep setup: 2 models x 2 quant x 2 shot types = 8 configs


## 3. Prompts

In [4]:
ZERO_SHOT_PROMPT = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a scientific research assistant. Summarize the following paper source in exactly one paragraph (100-150 words).
Use ONLY information from the source. Be concise and professional.<|eot_id|><|start_header_id|>user<|end_header_id|>
SOURCE TEXT:
{source}

KEY CONCEPTS TO INTEGRATE:
{keyphrases}

Write a faithful summary:<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

FEW_SHOT_PROMPT = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a scientific research assistant. Summarize the paper source using the provided key concepts.

Example:
Source: ... [previous text] ...
Key Concepts: - Convolutional Neural Networks for image classification. - Dropout layers to prevent overfitting.
Summary: This study explores the application of Convolutional Neural Networks (CNNs) in the domain of image recognition. To address the common issue of overfitting, the authors implement dropout layers, which improve generalization performance on standard benchmarks.

Task:
Source: {source}
Key Concepts: {keyphrases}
Write a faithful summary (100-150 words):<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

## 4. Helper Functions

In [5]:
def get_test_data(skip_samples, num_samples):
    print(f"  Loading test data (skipping {skip_samples})...")
    dataset = load_dataset("ccdv/arxiv-summarization", split="train", streaming=True)
    dataset = dataset.skip(skip_samples)
    
    test_data = []
    for entry in dataset:
        source = entry['article']
        summary = entry['abstract']
        
        if len(source) < 500 or len(summary) < 50 or len(source) > 15000:
            continue
            
        test_data.append({"source": source, "reference": summary})
        
        if len(test_data) >= num_samples:
            break
    
    print(f"  Test data ready: {len(test_data)} samples")
    return test_data

def load_sigext_model(model_id):
    print(f"  Loading SigExt: {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForTokenClassification.from_pretrained(model_id).to("cuda")
    return model, tokenizer

def load_llm(model_id, quant_config):
    quant_name = "4-bit" if "load_in_4bit" in quant_config else "8-bit"
    print(f"  Loading LLM ({quant_name}): {model_id}...")
    
    bnb_config = BitsAndBytesConfig(**quant_config)
    
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token
    
    return model, tokenizer

def extract_salient_sentences(text, model, tokenizer, max_length):
    sentences = [sent.text.strip() for sent in nlp(text).sents if len(sent.text.strip()) > 20]
    if not sentences: return [], ""
    
    salient_sentences = []
    for sent in sentences:
        inputs = tokenizer(sent, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
        with torch.no_grad():
            logits = model(**inputs).logits
        preds = torch.argmax(logits, dim=2)[0].tolist()
        valid_preds = preds[1:-1] if len(preds) > 2 else preds
        if valid_preds and (sum(valid_preds) / len(valid_preds)) > 0.5:
            salient_sentences.append(sent)
    
    keyphrases_text = "\n".join(f"- {s}" for s in salient_sentences)
    return salient_sentences, keyphrases_text

def preprocess_dataset(test_data, sigext_model, sigext_tokenizer, max_length):
    processed_data = []
    for item in tqdm(test_data, desc="    Extracting Salient Sentences"):
        salient_sents, keys_text = extract_salient_sentences(item['source'], sigext_model, sigext_tokenizer, max_length)
        processed_data.append({
            "source": item['source'],
            "reference": item['reference'],
            "salient_sentences": salient_sents,
            "keyphrases": keys_text
        })
    return processed_data

def create_chain(model, tokenizer, shot_type):
    gen_pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,
        temperature=0.1
    )
    llm = HuggingFacePipeline(pipeline=gen_pipe)
    prompt_tmpl = ZERO_SHOT_PROMPT if shot_type == "zero-shot" else FEW_SHOT_PROMPT
    prompt = PromptTemplate(template=prompt_tmpl, input_variables=["source", "keyphrases"])
    return prompt | llm | StrOutputParser()

def run_evaluation(processed_data, chain):
    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)
    metrics = {"bert": [], "rouge": [], "kir": []}
    samples = []
    
    for item in tqdm(processed_data, desc="    Generating & Evaluating"):
        res = chain.invoke({"source": item['source'], "keyphrases": item['keyphrases']})
        gen_summary = res.split("assistant<|end_header_id|>")[-1].strip()
        
        r_scores = scorer.score(item['reference'], gen_summary)
        metrics["rouge"].append(r_scores['rouge1'].fmeasure)
        
        _, _, F1 = bert_score([gen_summary], [item['reference']], lang="en", verbose=False)
        metrics["bert"].append(F1.mean().item())
        
        hits = 0
        if item['salient_sentences']:
            gen_lower = gen_summary.lower()
            for sent in item['salient_sentences']:
                words = [w.lower() for w in sent.split() if len(w) > 4]
                if words and sum(1 for w in words if w in gen_lower) / len(words) > 0.3:
                    hits += 1
            metrics["kir"].append(hits / len(item['salient_sentences']))
        else:
            metrics["kir"].append(0.0)
            
        samples.append({
            "source": item['source'][:500] + "...",
            "reference": item['reference'],
            "generated": gen_summary,
            "scores": {"bert": metrics["bert"][-1], "rouge1": metrics["rouge"][-1], "kir": metrics["kir"][-1]}
        })
    return metrics, samples

def save_results(metrics, samples, config_name, quant_name, shot_type, sigext_id, llm_id, num_samples, skip_samples, output_dir):
    results = {
        "run_info": {
            "timestamp": datetime.now().isoformat(),
            "config": config_name,
            "quantization": quant_name,
            "inference_type": shot_type,
            "sigext_model": sigext_id,
            "llm_model": llm_id,
            "num_test_samples": num_samples,
            "skip_train_samples": skip_samples
        },
        "metrics": {
            "bert_score": {"mean": float(np.mean(metrics['bert'])), "std": float(np.std(metrics['bert']))},
            "rouge1": {"mean": float(np.mean(metrics['rouge'])), "std": float(np.std(metrics['rouge']))},
            "kir": {"mean": float(np.mean(metrics['kir'])), "std": float(np.std(metrics['kir']))}
        },
        "raw_scores": metrics,
        "samples": samples
    }
    filename = f"results_{shot_type.replace('-', '_')}_{quant_name}_{config_name}.json"
    filepath = os.path.join(output_dir, filename)
    with open(filepath, 'w') as f: json.dump(results, f, indent=2, ensure_ascii=False)
    return filepath, results

def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

## 5. Sweep Execution

In [ ]:
all_results = []
PREPROCESSED_DATA = {}

print("=" * 60)
print("PHASE 1: PRE-PROCESSING")
print("=" * 60)

for config_name, sigext_config in SIGEXT_CONFIGS.items():
    print(f"\n  --- Config: {config_name} ---")
    sigext_model, sigext_tokenizer = load_sigext_model(sigext_config["model_id"])
    test_data = get_test_data(sigext_config["skip_samples"], GLOBAL_CONFIG["num_test_samples"])
    PREPROCESSED_DATA[config_name] = preprocess_dataset(test_data, sigext_model, sigext_tokenizer, 1024)
    del sigext_model, sigext_tokenizer
    clear_gpu_memory()

print("\n" + "=" * 60)
print("PHASE 2: INFERENCE SWEEP")
print("=" * 60)

for quant_name, quant_config in QUANT_CONFIGS.items():
    print(f"\n  >> Quantization: {quant_name}")
    llm_model, llm_tokenizer = load_llm(GLOBAL_CONFIG["llm_model_id"], quant_config)
    
    for config_name, _ in SIGEXT_CONFIGS.items():
        processed_data = PREPROCESSED_DATA[config_name]
        for shot_type in SHOT_TYPES:
            print(f"    Runing {shot_type} for {config_name}...")
            chain = create_chain(llm_model, llm_tokenizer, shot_type)
            metrics, samples = run_evaluation(processed_data, chain)
            filepath, results = save_results(metrics, samples, config_name, quant_name, shot_type, 
                                          SIGEXT_CONFIGS[config_name]["model_id"], GLOBAL_CONFIG["llm_model_id"], 
                                          len(samples), SIGEXT_CONFIGS[config_name]["skip_samples"], GLOBAL_CONFIG["output_dir"])
            all_results.append({"config": config_name, "quant": quant_name, "shot": shot_type, 
                               "bert": results["metrics"]["bert_score"]["mean"], 
                               "rouge": results["metrics"]["rouge1"]["mean"], 
                               "kir": results["metrics"]["kir"]["mean"]})
            del chain
            gc.collect()
            
    del llm_model, llm_tokenizer
    clear_gpu_memory()

print("\n" + "=" * 60)
print("SWEEP COMPLETE!")
print("=" * 60)

PHASE 1: PRE-PROCESSING

  --- Config: 1k-060t ---
  Loading SigExt: LookUpMark/sigext-arxiv-en-1k-060t...
  Loading test data (skipping 10000)...


'HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.' thrown while requesting GET https://huggingface.co/datasets/ccdv/arxiv-summarization/resolve/240aaf1a969b3f8cd0ade6986bfad0cd730ee288/section/train-00000-of-00015.parquet
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: b1def2da-5026-4a3a-a7ef-3cabb91e6732)')' thrown while requesting GET https://huggingface.co/datasets/ccdv/arxiv-summarization/resolve/240aaf1a969b3f8cd0ade6986bfad0cd730ee288/section/train-00000-of-00015.parquet
Retrying in 2s [Retry 2/5].
'HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.' thrown while requesting GET https://huggingface.co/datasets/ccdv/arxiv-summarization/resolve/240aaf1a969b3f8cd0ade6986bfad0cd730ee288/section/train-00000-of-00015.parquet
Retrying in 4s [Retry 3/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): R

  Test data ready: 100 samples


    Extracting Salient Sentences: 100%|██████████| 100/100 [08:30<00:00,  5.10s/it]



  --- Config: 5k-060t ---
  Loading SigExt: LookUpMark/sigext-arxiv-en-5k-060t...


## 6. Summary Table

In [ ]:
print("=" * 80)
print(f"{'Config':<10} {'Quant':<6} {'Shot':<10} {'BERT':<8} {'ROUGE-1':<8} {'KIR':<8}")
print("-" * 80)
for r in sorted(all_results, key=lambda x: (x['config'], x['quant'], x['shot'])):
    print(f"{r['config']:<10} {r['quant']:<6} {r['shot']:<10} {r['bert']:.4f}   {r['rouge']:.4f}   {r['kir']:.2%}")

summary_file = os.path.join(GLOBAL_CONFIG["output_dir"], "summary.json")
with open(summary_file, 'w') as f: json.dump(all_results, f, indent=2)
print(f"\nSummary saved to: {summary_file}")